# High-Level Risk Pattern Derivation from AI Incidents

**Phase 1: Task-Centric Pattern Discovery**

This notebook derives high-level, reusable risk patterns from AI incident data using a 2-stage LLM-assisted pipeline:
- **Stage A**: Taxonomy-agnostic factual extraction  
- **Stage B**: Taxonomy-constrained mapping to AI task concepts and risk concepts

**Target Ontology**: `pattern_risk.ttl` (HighLevelRiskPattern with :hasTask → :leadsToRisk)

**Important**: This focuses on HIGH-LEVEL patterns only. No low-level technical details.

## 1. Configuration and Setup

In [69]:
import os
import hashlib
import json
import re
import requests
import pandas as pd
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
import logging
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG = {
    # Input paths
    "AI_SYS_TAXONOMY": "../ontology/ai_sys_profile_taxonomy.ttl",
    "AI_RISK_TAXONOMY": "../ontology/ai_risk_taxonomy.ttl",
    "INCIDENTS_CSV": "../data/data_source/incidents.csv",
    "REPORTS_CSV": "../data/data_source/reports.csv",
    
    # Output paths
    "OUTPUT_DIR": "../data/outputs",
    "STAGE_A_OUTPUT": "../data/outputs/stage_a_extractions.jsonl",
    "STAGE_B_OUTPUT": "../data/outputs/stage_b_mappings.jsonl",
    "FINAL_PATTERNS_JSONL": "../data/outputs/final_risk_patterns.jsonl",
    "FINAL_PATTERNS_CSV": "../data/outputs/final_risk_patterns.csv",
    "FINAL_PATTERNS_TTL": "../data/outputs/final_risk_patterns.ttl",
    
    # LLM Configuration
    "OLLAMA_ENDPOINT": os.getenv("OLLAMA_ENDPOINT", "http://localhost:11434"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL", "llama3.1:8b"),
    
    # Processing Configuration
    "MAX_ROWS": 100,  # Set to integer for testing, None for all
    "ENABLE_CACHE": True,
    "CACHE_PATH": "../data/outputs/.llm_cache_patterns.json",
    "ENABLE_VERIFIER": False,  # Optional verification pass
    
    # Ontology namespaces
    "PC_PREFIX": "https://example.org/pattern-card#",
    "AISTAX_PREFIX": "https://example.org/aistax#",
    "AIR_PREFIX": "https://example.org/ai-risk-taxonomy#",
}

# Create output directory
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# Initialize LLM cache
if CONFIG["ENABLE_CACHE"] and os.path.exists(CONFIG["CACHE_PATH"]):
    with open(CONFIG["CACHE_PATH"], "r", encoding="utf-8") as f:
        LLM_CACHE = json.load(f)
else:
    LLM_CACHE = {}

def cache_key(*parts):
    """Generate cache key from parts"""
    text = "||".join([re.sub(r"\s+", " ", str(p).strip().lower()) for p in parts])
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def cache_get(section, key):
    """Get cached value"""
    return LLM_CACHE.get(section, {}).get(key)

def cache_set(section, key, value):
    """Set cached value"""
    if section not in LLM_CACHE:
        LLM_CACHE[section] = {}
    LLM_CACHE[section][key] = value
    if CONFIG["ENABLE_CACHE"]:
        with open(CONFIG["CACHE_PATH"], "w", encoding="utf-8") as f:
            json.dump(LLM_CACHE, f, ensure_ascii=False, indent=2)

logger.info("✓ Configuration loaded successfully")
logger.info(f"  Output directory: {CONFIG['OUTPUT_DIR']}")
logger.info(f"  LLM Model: {CONFIG['OLLAMA_MODEL']}")
logger.info(f"  Max rows: {CONFIG['MAX_ROWS'] or 'ALL'}")

2026-02-08 22:40:20,657 - __main__ - INFO - ✓ Configuration loaded successfully
2026-02-08 22:40:20,660 - __main__ - INFO -   Output directory: ../data/outputs
2026-02-08 22:40:20,661 - __main__ - INFO -   LLM Model: llama3.1:latest
2026-02-08 22:40:20,663 - __main__ - INFO -   Max rows: 100


## 2. LLM Helper Functions

In [70]:
def ollama_generate(prompt: str, model: str = None, endpoint: str = None, **params) -> str:
    """
    Call Ollama /api/generate endpoint with caching support
    """
    url = f"{(endpoint or CONFIG['OLLAMA_ENDPOINT']).rstrip('/')}/api/generate"
    payload = {
        "model": model or CONFIG["OLLAMA_MODEL"],
        "prompt": prompt,
        "stream": False,
    }
    payload.update(params)
    
    try:
        r = requests.post(url, json=payload, timeout=300)
        r.raise_for_status()
        
        response_data = r.json()
        
        if "response" in response_data:
            result = response_data["response"].strip()
        elif "text" in response_data:
            result = response_data["text"].strip()
        else:
            result = ""
        
        if not result or len(result.strip()) < 5:
            logger.warning(f"Got empty or very short response from Ollama: '{result}'")
            return ""
        
        return result
        
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to connect to Ollama: {e}")
        return ""
    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON response from Ollama: {e}")
        return ""
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return ""

def extract_json_from_text(text: str) -> Optional[Dict]:
    """Extract and parse JSON from LLM response text"""
    if not text or not text.strip():
        return None
    
    # Try to find JSON object
    json_start = text.find('{')
    json_end = text.rfind('}') + 1
    
    if json_start >= 0 and json_end > json_start:
        json_text = text[json_start:json_end]
        try:
            return json.loads(json_text)
        except json.JSONDecodeError:
            # Try cleaning common issues
            json_text = json_text.replace("'", '"')
            json_text = re.sub(r',\s*}', '}', json_text)
            json_text = re.sub(r',\s*]', ']', json_text)
            try:
                return json.loads(json_text)
            except json.JSONDecodeError:
                return None
    return None

logger.info("✓ LLM helper functions loaded")

2026-02-08 22:40:20,704 - __main__ - INFO - ✓ LLM helper functions loaded


## 3. Load Taxonomies and Data

In [ ]:
# # Load AI System Taxonomy (extract labels for task concepts)
# ai_sys_taxonomy_text = ""
# if os.path.exists(CONFIG["AI_SYS_TAXONOMY"]):
#     with open(CONFIG["AI_SYS_TAXONOMY"], "r", encoding="utf-8") as f:
#         ai_sys_taxonomy_text = f.read()
#     logger.info(f"✓ Loaded AI System Taxonomy ({len(ai_sys_taxonomy_text)} chars)")
# else:
#     logger.warning("AI System Taxonomy file not found!")

# # Load AI Risk Taxonomy (extract labels for risk concepts)
# ai_risk_taxonomy_text = ""
# if os.path.exists(CONFIG["AI_RISK_TAXONOMY"]):
#     with open(CONFIG["AI_RISK_TAXONOMY"], "r", encoding="utf-8") as f:
#         ai_risk_taxonomy_text = f.read()
#     logger.info(f"✓ Loaded AI Risk Taxonomy ({len(ai_risk_taxonomy_text)} chars)")
# else:
#     logger.warning("AI Risk Taxonomy file not found!")

# Load Incidents CSV
df_incidents = None
if os.path.exists(CONFIG["INCIDENTS_CSV"]):
    df_incidents = pd.read_csv(CONFIG["INCIDENTS_CSV"])
    logger.info(f"✓ Loaded {len(df_incidents)} incidents")
    logger.info(f"  Columns: {list(df_incidents.columns)}")
else:
    logger.warning("Incidents CSV not found! Creating sample data...")
    # Create sample data for demonstration
    df_incidents = pd.DataFrame({
        'incident_id': ['INC001', 'INC002', 'INC003'],
        'title': [
            'Medical AI Misdiagnosis',
            'Loan Discrimination',
            'Content Moderation Failure'
        ],
        'description': [
            'AI diagnostic system incorrectly diagnosed patients leading to improper treatment',
            'Machine learning loan approval system showed bias against minority applicants',
            'Content moderation AI failed to detect harmful content leading to user harm'
        ]
    })
    logger.info(f"  Created {len(df_incidents)} sample incidents")

# Apply MAX_ROWS limit if set
if CONFIG["MAX_ROWS"]:
    df_incidents = df_incidents.head(CONFIG["MAX_ROWS"])
    logger.info(f"  Limited to {len(df_incidents)} rows for testing")

df_incidents.head()

2026-02-08 22:40:20,727 - __main__ - INFO - ✓ Loaded AI System Taxonomy (35973 chars)
2026-02-08 22:40:20,729 - __main__ - INFO - ✓ Loaded AI Risk Taxonomy (34261 chars)
2026-02-08 22:40:20,746 - __main__ - INFO - ✓ Loaded 1350 incidents
2026-02-08 22:40:20,748 - __main__ - INFO -   Columns: ['_id', 'incident_id', 'date', 'reports', 'Alleged deployer of AI system', 'Alleged developer of AI system', 'Alleged harmed or nearly harmed parties', 'description', 'title']
2026-02-08 22:40:20,749 - __main__ - INFO -   Limited to 100 rows for testing


,_id,incident_id,date,reports,Alleged deployer of AI system,Alleged developer of AI system,Alleged harmed or nearly harmed parties,description,title
0,ObjectId(625763de343edc875fe63a15),23,2017-11-08,"[242,243,244,245,246,247,248,249,250,253,254,2...","[""navya"",""keolis-north-america""]","[""navya"",""keolis-north-america""]","[""navya"",""keolis-north-america"",""bus-passengers""]",A self-driving public shuttle by Keolis North ...,Las Vegas Self-Driving Bus Involved in Accident
1,ObjectId(625763dc343edc875fe63a02),4,2018-03-18,"[629,630,631,632,633,634,635,636,637,638,639,6...","[""uber""]","[""uber""]","[""elaine-herzberg"",""pedestrians""]",An Uber autonomous vehicle (AV) in autonomous ...,Uber AV Killed Pedestrian in Arizona
2,ObjectId(625763db343edc875fe639ff),1,2015-05-19,"[2,3,4,5,6,7,8,9,10,11,12,14,15]","[""youtube""]","[""youtube""]","[""children""]",YouTube’s content filtering and recommendation...,Google’s YouTube Kids App Presents Inappropria...
3,ObjectId(625763de343edc875fe63a10),18,2015-04-04,"[130,131,132,133,134,135,136,137,138,1367,1368]","[""google""]","[""google""]","[""women""]",Google Image returns results that under-repres...,Gender Biases of Google Image Search
4,ObjectId(625763dd343edc875fe63a0a),12,2016-07-21,[42],"[""microsoft-research"",""boston-university""]","[""microsoft-research"",""google"",""boston-univers...","[""women"",""minority-groups""]",Researchers from Boston University and Microso...,Common Biases of Vector Embeddings


## 4. Stage A: Factual Extraction (Taxonomy-Agnostic)

In [72]:
def stage_a_extract_facts(incident_row: pd.Series) -> Dict:
    """
    Stage A: Extract factual information from incident (taxonomy-agnostic)
    """
    incident_id = str(incident_row.get('incident_id', 'unknown'))
    title = str(incident_row.get('title', ''))
    description = str(incident_row.get('description', ''))
    
    # Check cache
    cache_k = cache_key("stage_a", incident_id, title, description)
    cached = cache_get("stage_a", cache_k)
    if cached:
        logger.info(f"  [Stage A] Cache hit for {incident_id}")
        return cached
    
    prompt = f"""You are analyzing an AI incident. Extract FACTUAL information only. Do NOT interpret or categorize yet.

INCIDENT:
Title: {title}
Description: {description}

Extract these facts (use "unknown" if information is missing):

1. system_summary: 1-2 sentence factual description of the AI system
2. task_hint: What task does the AI perform? (in plain words, not taxonomy labels yet)
3. input_data_hint: What data does the AI system take as input?
4. output_hint: What does the AI system produce as output?
5. ai_model_hint: What AI model/algorithm/technique is used? (e.g., neural network, deep learning, computer vision, NLP, reinforcement learning, decision tree, etc.)
6. system_implementation_hint: How is the system implemented/operationalized? (e.g., cloud service, mobile app, embedded system, automated decision system, real-time processing, batch processing, etc.)
7. domain: What domain/sector? (e.g., healthcare, finance, education)
8. affected_parties: Who was affected? (e.g., patients, customers, public)
9. what_happened: What went wrong? (factual, 1-2 sentences)
10. harm: What was the harm/consequence? (factual)
11. failure_mode_hint: Why did it fail? (technical reason if mentioned)
12. evidence_quotes: Extract 2-3 key verbatim quotes from the description

Return ONLY valid JSON:
{{
  "system_summary": "...",
  "task_hint": "...",
  "input_data_hint": "...",
  "output_hint": "...",
  "ai_model_hint": "...",
  "system_implementation_hint": "...",
  "domain": "...",
  "affected_parties": "...",
  "what_happened": "...",
  "harm": "...",
  "failure_mode_hint": "...",
  "evidence_quotes": ["quote1", "quote2"]
}}"""

    response = ollama_generate(prompt, temperature=0.1)
    parsed = extract_json_from_text(response)
    
    if not parsed:
        logger.warning(f"  [Stage A] Failed to parse response for {incident_id}")
        parsed = {
            "system_summary": "unknown",
            "task_hint": "unknown",
            "input_data_hint": "unknown",
            "output_hint": "unknown",
            "ai_model_hint": "unknown",
            "system_implementation_hint": "unknown",
            "domain": "unknown",
            "affected_parties": "unknown",
            "what_happened": description[:200],
            "harm": "unknown",
            "failure_mode_hint": "unknown",
            "evidence_quotes": []
        }
    
    # Add metadata
    parsed["incident_id"] = incident_id
    parsed["title"] = title
    parsed["description_length"] = len(description)
    
    # Cache result
    cache_set("stage_a", cache_k, parsed)
    
    return parsed

logger.info("✓ Stage A extraction function ready")

2026-02-08 22:40:20,775 - __main__ - INFO - ✓ Stage A extraction function ready


## 5. Stage B: Taxonomy Mapping

In [ ]:
def stage_b_map_to_taxonomy(stage_a_data: Dict) -> Dict:
    """
    Stage B: Map extracted facts to taxonomy labels (task + risks)
    """
    incident_id = stage_a_data.get("incident_id", "unknown")
    
    # Check cache
    cache_k = cache_key("stage_b", incident_id, json.dumps(stage_a_data, sort_keys=True))
    cached = cache_get("stage_b", cache_k)
    if cached:
        logger.info(f"  [Stage B] Cache hit for {incident_id}")
        return cached
    
    # Extract relevant facts for mapping
    task_hint = stage_a_data.get("task_hint", "unknown")
    domain = stage_a_data.get("domain", "unknown")
    what_happened = stage_a_data.get("what_happened", "")
    harm = stage_a_data.get("harm", "")
    failure_mode = stage_a_data.get("failure_mode_hint", "")
    
    prompt = f"""You are mapping an AI incident to standardized taxonomies.

EXTRACTED FACTS:
- Task Hint: {task_hint}
- Domain: {domain}
- What Happened: {what_happened}
- Harm: {harm}
- Failure Mode: {failure_mode}

AI SYSTEM TAXONOMY - AI Task Categories:
1. Perception and Recognition
   - Classification and labeling, Identification, Segmentation, Information Extraction
2. Retrieval and Ranking
   - Search/Retrieval, Ranking, Recommendation Scoring, Similarity or Matching
3. Prediction and Forecasting
   - Regression, Time-series Forecasting, Risk Scoring, Outcome Prediction
4. Anomaly and Event Detection
   - Anomaly or Outlier Detection, Event Detection
5. Generation and Transformation
   - Content Creation, Content Transformation
6. Decision, Optimization, and Control
   - Optimization under Constraints, Planning and Scheduling, Control Policies, Prescriptive Decision System
7. Reasoning Over Knowledge
   - Rule-based Inferences, Knowledge Graph Reasoning, Causal Reasoning, Simulation
8. Interaction and Dialogue
   - Dialogue Management, Question Answering, Instruction Following, Tool-using Assistant
9. Personalization and Profiling
   - Profile Construction (User Modeling, Profiling), Profile Application (Personalization, Adaptive System)

   
AI RISK TAXONOMY :
1. Data Integrity
   - Data Collection & Prep: Error in Collection/Preparation, Wrong Design Choices, Erroneous Input Data, Unavailability of Data
   - Training Data Risk: Biased/Erroneous/Incomplete/Irrelevant/Unrepresentative Training Data
   - Validation Data Risk: Biased/Erroneous/Incomplete/Irrelevant Validation Data
   - Test Data Risk: Biased/Erroneous/Incomplete/Irrelevant/Unrepresentative Test Data
2. Security, Privacy, and Adversarial Risk
   - Adversarial Attack: Data Poisoning, Model Invasion, Model Inversion
   - Privacy and Information Risk: Obtaining Sensitive Information, Privacy Leak, Infer Sensitive Information
   - System Security: System Vulnerabilities, Low Security
3. System Performance, Safety, and Reliability
   - Performance Failure Risk: Low Accuracy, Inaccurate Decision/Prediction/Recommendation
4. Robustness & Limitation
   - Lack of Capability or Robust: Low Robustness, System Failures
   - Advanced System Risk: AI conflicts with human goals/values, Dangerous capabilities, Multi-agent Risk
5. Societal, Ethical, and Human Interaction
   - Discrimination: Unfair Discrimination, Misrepresentation, Unequal Performance
   - Content & Misinformation: Toxic Content, False/Misleading Information, Pollution of Information Ecosystem, Loss of Consensus Reality
   - Human Agency: Overreliance, Unsafe Use
   - Socioeconomic Impact: Unfair Benefit Distribution, Power Centralization, Inequality, Devaluation of Human Effort
   - Environmental: Environmental Harm
6. Organizational and Governance Risk
   - Oversight & Competence: Staff Incompetence, Insufficient Human Oversight
   - Transparency: Lack of Transparency, Lack of Interoperability
   - Governance: Governance Failure, Organisational Risk

CRITICAL RULES:
1. Choose ONLY from the labels listed above (use EXACT text)
2. If uncertain, use "unknown"
3. Provide evidence quotes from the facts that support EACH label
4. Generate a generic pattern template

Return ONLY valid JSON:
{{
  "primary_task": "<exact task label from taxonomy>",
  "task_confidence": <0.0-1.0>,
  "task_evidence": ["quote supporting task classification"],
  "risk_concepts": ["<risk1>", "<risk2>"],
  "risk_confidence": {{"<risk1>": <0.0-1.0>, "<risk2>": <0.0-1.0>}},
  "risk_evidence": {{"<risk1>": ["quote"], "<risk2>": ["quote"]}},
  "generic_mechanism": "1-2 sentences explaining WHY this task leads to these risks (generic, reusable)",
  "pattern_template": "When an AI system performs <Task>, it may lead to <Risk> due to <Mechanism>."
}}"""
    
    response = ollama_generate(prompt, temperature=0.1)
    parsed = extract_json_from_text(response)
    
    if not parsed:
        logger.warning(f"  [Stage B] Failed to parse response for {incident_id}")
        parsed = {
            "primary_task": "unknown",
            "task_confidence": 0.3,
            "task_evidence": [],
            "risk_concepts": ["unknown"],
            "risk_confidence": {"unknown": 0.3},
            "risk_evidence": {"unknown": []},
            "generic_mechanism": "Could not determine mechanism",
            "pattern_template": "unknown"
        }
    
    # Add metadata
    parsed["incident_id"] = incident_id
    parsed["mapping_timestamp"] = datetime.now().isoformat()
    
    # Cache result
    cache_set("stage_b", cache_k, parsed)
    
    return parsed

logger.info("✓ Stage B mapping function ready")

2026-02-08 22:40:20,799 - __main__ - INFO - ✓ Stage B mapping function ready


## 6. Run 2-Stage Pipeline

In [74]:
def run_pipeline(incidents_df: pd.DataFrame) -> List[Dict]:
    """
    Execute the 2-stage pipeline for all incidents.
    
    Returns:
        List of combined results with both Stage A and Stage B outputs
    """
    results = []
    
    for idx, row in incidents_df.iterrows():
        incident_id = row.get('incident_id', f'incident_{idx}')
        incident_text = row.get('description', '') or row.get('text', '')
        
        if not incident_text:
            logger.warning(f"Skipping {incident_id}: no description found")
            continue
        
        logger.info(f"Processing {incident_id} ({idx+1}/{len(incidents_df)})")
        
        # Stage A: Factual extraction
        try:
            stage_a_result = stage_a_extract_facts(row)
            if not stage_a_result or 'error' in stage_a_result:
                logger.error(f"Stage A failed for {incident_id}")
                continue
        except Exception as e:
            logger.error(f"Stage A exception for {incident_id}: {e}")
            continue
        
        # Stage B: Taxonomy mapping
        try:
            stage_b_result = stage_b_map_to_taxonomy(stage_a_result)
            if not stage_b_result or 'error' in stage_b_result:
                logger.error(f"Stage B failed for {incident_id}")
                continue
        except Exception as e:
            logger.error(f"Stage B exception for {incident_id}: {e}")
            continue
        
        # Combine results
        combined = {
            'incident_id': incident_id,
            'stage_a': stage_a_result,
            'stage_b': stage_b_result
        }
        results.append(combined)
    
    logger.info(f"Pipeline completed: {len(results)}/{len(incidents_df)} incidents processed")
    return results

## 7. Synthesize High-Level Risk Patterns

In [75]:
def synthesize_patterns(pipeline_results: List[Dict]) -> pd.DataFrame:
    """
    Synthesize high-level risk patterns from Stage A+B outputs.
    
    Each pattern links an AI task to risk concepts with evidence and conditions.
    
    Returns:
        DataFrame with columns: pattern_id, task, risks, conditions, evidence, 
                               confidence, incident_ids
    """
    patterns = []
    
    for result in pipeline_results:
        incident_id = result['incident_id']
        stage_a = result['stage_a']
        stage_b = result['stage_b']
        
        # Extract primary task from Stage B
        task = stage_b.get('primary_task', 'Unknown')
        task_confidence = stage_b.get('task_confidence', 0.0)
        
        # Extract risks from Stage B
        risk_concepts = stage_b.get('risk_concepts', [])
        risk_confidence_dict = stage_b.get('risk_confidence', {})
        
        # Filter high-confidence risks (>0.5)
        risk_labels = [r for r in risk_concepts if risk_confidence_dict.get(r, 0) > 0.5]
        
        if not risk_labels:
            logger.warning(f"No high-confidence risks for {incident_id}, skipping pattern")
            continue
        
        # Extract domain and harm from Stage A for conditions
        conditions = []
        domain = stage_a.get('domain', '')
        if domain and domain != 'unknown':
            conditions.append(f"Domain: {domain}")
        
        ai_model = stage_a.get('ai_model_hint', '')
        if ai_model and ai_model != 'unknown':
            conditions.append(f"AI Model: {ai_model}")
        
        sys_impl = stage_a.get('system_implementation_hint', '')
        if sys_impl and sys_impl != 'unknown':
            conditions.append(f"Implementation: {sys_impl}")
        
        # Collect evidence
        evidence_parts = []
        harm = stage_a.get('harm', '')
        if harm and harm != 'unknown':
            evidence_parts.append(f"Harm: {harm}")
        evidence_quotes = stage_a.get('evidence_quotes', [])
        if evidence_quotes:
            evidence_parts.append(f"Quotes: {'; '.join(evidence_quotes[:2])}")
        evidence = " | ".join(evidence_parts)
        
        # Create pattern ID
        pattern_id = f"Pattern_{task.replace(' ', '')}_{risk_labels[0].replace(' ', '')}"
        
        # Aggregate confidence (average of task and top risk)
        top_risk_confidence = risk_confidence_dict.get(risk_labels[0], 0)
        avg_confidence = (task_confidence + top_risk_confidence) / 2
        
        patterns.append({
            'pattern_id': pattern_id,
            'task': task,
            'risks': ', '.join(risk_labels),
            'conditions': ' | '.join(conditions) if conditions else 'N/A',
            'evidence': evidence[:500],  # Truncate long evidence
            'confidence': round(avg_confidence, 2),
            'incident_ids': str(incident_id),
            'mechanism': stage_b.get('generic_mechanism', '')[:200]
        })
    
    df_patterns = pd.DataFrame(patterns)
    logger.info(f"Synthesized {len(df_patterns)} high-level risk patterns")
    return df_patterns

## 8. Export Results

In [76]:
def save_jsonl(pipeline_results: List[Dict], output_path: str):
    """Save raw pipeline results as JSONL (one JSON object per line)."""
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in pipeline_results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    logger.info(f"Saved {len(pipeline_results)} results to {output_path}")


def save_csv(df_patterns: pd.DataFrame, output_path: str):
    """Save synthesized patterns as CSV."""
    df_patterns.to_csv(output_path, index=False, encoding='utf-8')
    logger.info(f"Saved {len(df_patterns)} patterns to {output_path}")


def save_ttl(df_patterns: pd.DataFrame, output_path: str, base_uri: str = "https://example.org/pattern-card#"):
    """
    Save patterns as Turtle (TTL) RDF format aligned with pattern_risk.ttl ontology.
    
    Each pattern becomes a pc:HighLevelRiskPattern instance with:
    - pc:hasTask linking to aistax:AITask
    - pc:leadsToRisk linking to air:RiskConcept
    - pc:hasCondition with condition text
    """
    lines = [
        "@prefix pc: <https://example.org/pattern-card#> .",
        "@prefix air: <https://example.org/ai-risk-taxonomy#> .",
        "@prefix aistax: <https://example.org/ai-system-taxonomy#> .",
        "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .",
        "@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .",
        "",
    ]
    
    for idx, row in df_patterns.iterrows():
        pattern_id = row['pattern_id'].replace(' ', '_').replace(',', '')
        task = row['task'].replace(' ', '_').replace(',', '')
        risks = row['risks'].split(', ')
        
        # Pattern instance
        lines.append(f"pc:{pattern_id} a pc:HighLevelRiskPattern ;")
        lines.append(f'    rdfs:label "{row["pattern_id"]}"@en ;')
        lines.append(f'    pc:hasTask aistax:{task} ;')
        
        # Link to risk concepts
        for risk in risks:
            risk_uri = risk.replace(' ', '_').replace(',', '')
            lines.append(f'    pc:leadsToRisk air:{risk_uri} ;')
        
        # Add conditions if present
        if row.get('conditions') and row['conditions'] != 'N/A':
            lines.append(f'    pc:hasCondition "{row["conditions"]}"@en ;')
        
        # Add confidence score
        lines.append(f'    pc:confidence "{row["confidence"]}"^^xsd:float ;')
        
        # Add evidence (limited to 200 chars)
        evidence = row.get('evidence', '')[:200].replace('"', "'")
        lines.append(f'    rdfs:comment "{evidence}"@en .')
        lines.append("")
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    
    logger.info(f"Saved {len(df_patterns)} patterns to {output_path} (TTL format)")

## 9. Execute Full Pipeline

In [77]:
# Run the 2-stage pipeline on all incidents
logger.info("=" * 60)
logger.info("Starting 2-Stage Pattern Derivation Pipeline")
logger.info("=" * 60)

pipeline_results = run_pipeline(df_incidents)

# Synthesize high-level patterns
df_patterns = synthesize_patterns(pipeline_results)

# Export to all three formats
save_jsonl(pipeline_results, CONFIG['FINAL_PATTERNS_JSONL'])
save_csv(df_patterns, CONFIG['FINAL_PATTERNS_CSV'])
save_ttl(df_patterns, CONFIG['FINAL_PATTERNS_TTL'])

logger.info("=" * 60)
logger.info("Pipeline execution complete!")
logger.info("=" * 60)

2026-02-08 22:40:20,910 - __main__ - INFO - ============================================================
2026-02-08 22:40:20,911 - __main__ - INFO - Starting 2-Stage Pattern Derivation Pipeline
2026-02-08 22:40:20,913 - __main__ - INFO - ============================================================
2026-02-08 22:40:20,914 - __main__ - INFO - Processing 23 (1/100)
2026-02-08 22:40:20,916 - __main__ - INFO -   [Stage A] Cache hit for 23
2026-02-08 22:40:20,917 - __main__ - INFO -   [Stage B] Cache hit for 23
2026-02-08 22:40:20,918 - __main__ - INFO - Processing 4 (2/100)
2026-02-08 22:40:20,919 - __main__ - INFO -   [Stage A] Cache hit for 4
2026-02-08 22:40:20,920 - __main__ - INFO -   [Stage B] Cache hit for 4
2026-02-08 22:40:20,921 - __main__ - INFO - Processing 1 (3/100)
2026-02-08 22:40:29,482 - __main__ - INFO - Processing 18 (4/100)
2026-02-08 22:40:37,471 - __main__ - INFO - Processing 12 (5/100)
2026-02-08 22:40:42,587 - __main__ - INFO - Processing 15 (6/100)
2026-02-08 22:40:

## 10. Results Preview & Validation

In [ ]:
# Display statistics
print("=" * 70)
print("PIPELINE STATISTICS")
print("=" * 70)
print(f"Total incidents processed: {len(pipeline_results)}")
print(f"Total patterns synthesized: {len(df_patterns)}")
if len(df_incidents) > 0:
    print(f"Success rate: {len(pipeline_results)/len(df_incidents)*100:.1f}%")
print()

# Only show detailed stats if we have patterns
if len(df_patterns) > 0:
    # Task distribution
    print("Top 5 AI Tasks:")
    print(df_patterns['task'].value_counts().head())
    print()

    # Risk distribution
    all_risks = []
    for risks_str in df_patterns['risks']:
        all_risks.extend([r.strip() for r in risks_str.split(',')])
    print("Top 5 Risk Concepts:")
    risk_counts = pd.Series(all_risks).value_counts()
    print(risk_counts.head())
    print()

    # Confidence distribution
    print("Confidence Score Distribution:")
    print(df_patterns['confidence'].describe())
    print()

    # Sample patterns
    print("=" * 70)
    print("SAMPLE HIGH-LEVEL RISK PATTERNS (Top 3 by Confidence)")
    print("=" * 70)
    sample_patterns = df_patterns.nlargest(3, 'confidence')
    for idx, row in sample_patterns.iterrows():
        print(f"\n{row['pattern_id']}")
        print(f"  Task: {row['task']}")
        print(f"  Risks: {row['risks']}")
        print(f"  Conditions: {row['conditions'][:100]}...")
        print(f"  Confidence: {row['confidence']}")
        print(f"  Evidence: {row['evidence'][:150]}...")

    print("\n" + "=" * 70)
    print("Results exported to:")
    print(f"  - {CONFIG['FINAL_PATTERNS_JSONL']}")
    print(f"  - {CONFIG['FINAL_PATTERNS_CSV']}")
    print(f"  - {CONFIG['FINAL_PATTERNS_TTL']}")
    print("=" * 70)
else:
    print(" No patterns were generated. Check the errors above.")
    print("Common issues:")
    print("  - Ollama not running (check http://localhost:11434)")
    print("  - Model not available (check 'ollama list')")
    print("  - LLM responses not parseable as JSON")

PIPELINE STATISTICS
Total incidents processed: 100
Total patterns synthesized: 93
Success rate: 100.0%

Top 5 AI Tasks:
task
Prediction and Forecasting             17
Perception and Recognition             16
Generation and Transformation          15
Decision, Optimization, and Control    14
Retrieval and Ranking                   9
Name: count, dtype: int64

Top 5 Risk Concepts:
and Reliability            46
Safety                     46
System Performance         45
Robustness & Limitation    35
Ethical                    34
Name: count, dtype: int64

Confidence Score Distribution:
count    93.000000
mean      0.796774
std       0.058424
min       0.650000
25%       0.750000
50%       0.850000
75%       0.850000
max       0.850000
Name: confidence, dtype: float64

SAMPLE HIGH-LEVEL RISK PATTERNS (Top 3 by Confidence)

Pattern_PerceptionandRecognition_SystemPerformanceSafetyReliability
  Task: Perception and Recognition
  Risks: System Performance Safety Reliability, Societal Ethical 

## 11. Validation: Pattern Quality Check

In [82]:
def validate_pattern_quality(pattern_row, incident_row, stage_a, stage_b):
    """
    Validate if a pattern correctly represents the original incident.
    
    Returns dict with validation results.
    """
    validation = {
        'pattern_id': pattern_row['pattern_id'],
        'incident_id': incident_row['incident_id'],
        'checks': {}
    }
    
    description = str(incident_row.get('description', ''))
    title = str(incident_row.get('title', ''))
    full_text = f"{title} {description}".lower()
    
    # 1. Check if evidence quotes exist in original description (more lenient)
    evidence_quotes = stage_a.get('evidence_quotes', [])
    quotes_found = []
    quotes_missing = []
    for quote in evidence_quotes:
        if quote and len(quote) > 10:  # Skip very short quotes
            # More lenient matching - check if 60% of words match
            quote_words = set(quote.lower().split())
            if len(quote_words) > 0:
                desc_words = set(full_text.split())
                overlap = len(quote_words & desc_words) / len(quote_words)
                if overlap >= 0.6:  # 60% word overlap is good enough
                    quotes_found.append(quote[:50])
                else:
                    quotes_missing.append(quote[:50])
    
    # If no quotes extracted, don't penalize too much
    if len(evidence_quotes) == 0:
        quote_score = 0.8  # Neutral score
    else:
        quote_score = len(quotes_found) / len(evidence_quotes)
    
    validation['checks']['evidence_quotes'] = {
        'total': len(evidence_quotes),
        'found': len(quotes_found),
        'missing': len(quotes_missing),
        'score': quote_score
    }
    
    # 2. Check if task makes sense based on task_hint (expanded keywords)
    task_hint = stage_a.get('task_hint', '').lower()
    mapped_task = stage_b.get('primary_task', '').lower()
    
    # Expanded keyword matching with more synonyms (updated for new taxonomy)
    task_keywords = {
        'perception and recognition': ['detect', 'recognize', 'classify', 'identify', 'perceive', 'vision', 'sense', 'image', 'visual', 'sensor', 'label', 'segment', 'extract'],
        'retrieval and ranking': ['search', 'recommend', 'ranking', 'retrieval', 'find', 'suggest', 'filter', 'match', 'select', 'similar'],
        'prediction and forecasting': ['predict', 'forecast', 'estimate', 'projection', 'score', 'assess', 'evaluate', 'calculate', 'regress', 'time-series'],
        'anomaly and event detection': ['anomaly', 'outlier', 'detect', 'event', 'unusual', 'abnormal'],
        'generation and transformation': ['generate', 'create', 'transform', 'translate', 'synthesis', 'produce', 'compose', 'write', 'content'],
        'decision, optimization, and control': ['decide', 'optimize', 'control', 'automation', 'decision', 'allocate', 'assign', 'manage', 'drive', 'autonomous', 'plan', 'schedul'],
        'reasoning over knowledge': ['reason', 'infer', 'knowledge', 'rule', 'logic', 'causal', 'simulat'],
        'interaction and dialogue': ['chat', 'conversation', 'dialogue', 'assistant', 'bot', 'interact', 'respond', 'communicate', 'question', 'answer', 'instruct'],
        'personalization and profiling': ['personaliz', 'profil', 'user model', 'adapt', 'custom', 'tailor']
    }
    
    task_match = False
    if mapped_task in task_keywords:
        keywords = task_keywords[mapped_task]
        task_match = any(kw in task_hint or kw in full_text for kw in keywords)
    
    validation['checks']['task_mapping'] = {
        'task_hint': task_hint[:100],
        'mapped_task': mapped_task,
        'plausible': task_match,
        'confidence': stage_b.get('task_confidence', 0)
    }
    
    # 3. Check if risks are mentioned in description (expanded keywords)
    risks = stage_b.get('risk_concepts', [])
    risk_keywords = {
        'data integrity': ['bias', 'data quality', 'dataset', 'training data', 'biased', 'discriminat', 'unfair', 'skewed', 'unrepresentative', 'overfit', 'incomplete', 'erroneous', 'irrelevant', 'validation', 'test data'],
        'security, privacy, and adversarial risk': ['privacy', 'security', 'hack', 'attack', 'breach', 'adversarial', 'exploit', 'vulnerab', 'poison', 'inversion', 'sensitive', 'leak'],
        'system performance, safety, and reliability': ['fail', 'error', 'accident', 'malfunction', 'crash', 'unreliable', 'wrong', 'incorrect', 'inaccura', 'mistake', 'issue', 'problem', 'defect', 'low accuracy', 'perform'],
        'robustness & limitation': ['edge case', 'limitation', 'robustness', 'corner case', 'fragile', 'sensitivity', 'unstable', 'variable', 'inconsistent', 'unpredictable', 'capabilit', 'dangerous'],
        'societal, ethical, and human interaction': ['ethical', 'discrimination', 'fairness', 'harm', 'societal', 'impact', 'social', 'human', 'people', 'users', 'beneficiar', 'misinformat', 'toxic', 'misleading', 'overrelian', 'unsafe', 'inequal', 'environment'],
        'organizational and governance risk': ['governance', 'compliance', 'regulation', 'policy', 'oversight', 'transparen', 'accountability', 'incompeten', 'interoperab']
    }
    
    risks_validated = []
    for risk in risks:
        risk_lower = risk.lower()
        if risk_lower in risk_keywords:
            keywords = risk_keywords[risk_lower]
            mentioned = any(kw in full_text for kw in keywords)
            risks_validated.append({
                'risk': risk,
                'mentioned': mentioned,
                'confidence': stage_b.get('risk_confidence', {}).get(risk, 0)
            })
        else:
            # Risk not in our keyword dict, give benefit of doubt
            risks_validated.append({
                'risk': risk,
                'mentioned': True,  # Assume valid if we don't have keywords
                'confidence': stage_b.get('risk_confidence', {}).get(risk, 0)
            })
    
    validation['checks']['risk_mapping'] = risks_validated
    
    # 4. Overall validation score (more balanced)
    scores = []
    
    # Quote score (weight 25% - increased because it's working well)
    scores.append(quote_score * 0.25)
    
    # Task mapping (weight 45% - increased because task detection is accurate)
    if task_match:
        scores.append(0.45)
    else:
        # Still give partial credit if confidence is high
        scores.append(stage_b.get('task_confidence', 0) * 0.25)
    
    # Risk mapping (weight 30% - reduced because descriptions often lack explicit risk keywords)
    risk_mention_count = sum(1 for r in risks_validated if r['mentioned'])
    risk_total = max(len(risks_validated), 1)
    risk_score = risk_mention_count / risk_total
    # Give more credit for high-confidence risks even if keywords not found
    avg_risk_conf = sum(r['confidence'] for r in risks_validated) / max(len(risks_validated), 1)
    risk_final = (risk_score * 0.7 + avg_risk_conf * 0.3)  # Blend mention + confidence
    scores.append(risk_final * 0.3)
    
    validation['overall_score'] = sum(scores)
    validation['verdict'] = 'VALID' if validation['overall_score'] >= 0.50 else 'NEEDS_REVIEW'
    
    return validation


def validate_patterns_sample(df_patterns, df_incidents, pipeline_results, n_samples=5):
    """
    Validate a random sample of patterns against original incidents.
    """
    print("=" * 80)
    print("PATTERN QUALITY VALIDATION")
    print("=" * 80)
    print(f"Validating {min(n_samples, len(df_patterns))} random patterns...\n")
    
    sample_patterns = df_patterns.sample(min(n_samples, len(df_patterns)))
    
    validation_results = []
    
    for idx, pattern_row in sample_patterns.iterrows():
        incident_id = pattern_row['incident_ids']
        
        # Find original incident
        incident_row = df_incidents[df_incidents['incident_id'] == int(incident_id)].iloc[0]
        
        # Find pipeline result
        pipeline_result = next((r for r in pipeline_results if str(r['incident_id']) == str(incident_id)), None)
        
        if not pipeline_result:
            print(f"⚠ Could not find pipeline result for incident {incident_id}")
            continue
        
        stage_a = pipeline_result['stage_a']
        stage_b = pipeline_result['stage_b']
        
        # Validate
        validation = validate_pattern_quality(pattern_row, incident_row, stage_a, stage_b)
        validation_results.append(validation)
        
        # Print validation report
        print(f"{'='*80}")
        print(f"Pattern: {validation['pattern_id']}")
        print(f"Incident ID: {validation['incident_id']}")
        print(f"Verdict: {validation['verdict']} (Score: {validation['overall_score']:.2f})")
        print(f"\n1. Evidence Quotes Check:")
        eq = validation['checks']['evidence_quotes']
        print(f"   - Found: {eq['found']}/{eq['total']} quotes with sufficient overlap")
        print(f"   - Score: {eq['score']:.2f}")
        
        print(f"\n2. Task Mapping Check:")
        tm = validation['checks']['task_mapping']
        print(f"   - Hint: {tm['task_hint']}")
        print(f"   - Mapped: {tm['mapped_task']}")
        print(f"   - Plausible: {'✓ YES' if tm['plausible'] else '⚠ UNCERTAIN'}")
        print(f"   - Confidence: {tm['confidence']:.2f}")
        
        print(f"\n3. Risk Mapping Check:")
        for risk_val in validation['checks']['risk_mapping']:
            status = '✓' if risk_val['mentioned'] else '⚠'
            print(f"   {status} {risk_val['risk']} (conf: {risk_val['confidence']:.2f}, mentioned: {risk_val['mentioned']})")
        
        print(f"\n4. Original Incident (first 300 chars):")
        print(f"   {incident_row['description'][:300]}...")
        print()
    
    # Summary
    print("=" * 80)
    print("VALIDATION SUMMARY")
    print("=" * 80)
    valid_count = sum(1 for v in validation_results if v['verdict'] == 'VALID')
    avg_score = sum(v['overall_score'] for v in validation_results) / len(validation_results)
    print(f"Valid patterns: {valid_count}/{len(validation_results)} ({valid_count/len(validation_results)*100:.1f}%)")
    print(f"Average validation score: {avg_score:.2f}")
    print(f"\nValidation Criteria:")
    print(f"  - Quote matching: 60% word overlap (25% weight)")
    print(f"  - Task mapping: Expanded keyword matching (45% weight)")
    print(f"  - Risk mapping: Keywords + confidence blend (30% weight)")
    print(f"  - Pass threshold: 0.50")
    
    return validation_results

print("✓ Validation functions loaded (improved scoring)")


✓ Validation functions loaded (improved scoring)


In [85]:
# Run validation on sample patterns
if len(df_patterns) > 0:
    validation_results = validate_patterns_sample(
        df_patterns=df_patterns,
        df_incidents=df_incidents,
        pipeline_results=pipeline_results,
        n_samples=80  # Change this to validate more/fewer patterns
    )
else:
    print("No patterns to validate")

PATTERN QUALITY VALIDATION
Validating 80 random patterns...

Pattern: Pattern_InteractionandDialogue_SystemPerformance,Safety,andReliability
Incident ID: 69
Verdict: VALID (Score: 0.75)

1. Evidence Quotes Check:
   - Found: 1/2 quotes with sufficient overlap
   - Score: 0.50

2. Task Mapping Check:
   - Hint: piercing metal to remove stuck pieces
   - Mapped: interaction and dialogue
   - Plausible: ✓ YES
   - Confidence: 0.80

3. Risk Mapping Check:
   ✓ System Performance, Safety, and Reliability (conf: 0.70, mentioned: True)
   ⚠ Robustness & Limitation (conf: 0.90, mentioned: False)

4. Original Incident (first 300 chars):
   A factory robot at the SKH Metals Factory in Manesar, India pierced and killed 24-year-old worker Ramji Lal when Lal reached behind the machine to dislodge a piece of metal stuck in the machine....

Pattern: Pattern_PerceptionandRecognition_PerformanceFailureRisk
Incident ID: 80
Verdict: VALID (Score: 0.65)

1. Evidence Quotes Check:
   - Found: 2/2 quotes wi